# 02 — Data Cleaning | Retail Sales Analytics

**Goal:** validate the primary retail datasets, create useful date features, forward-fill missing values in the oil price series, verify store keys across datasets, and save the prepared results as Delta tables.

**Input:** the unchanged CSVs in `/Volumes/workspace/default/retail_raw` from the Kaggle competition.

**Output (Delta tables):** `workspace.default.sales_cleaned`, `workspace.default.stores_cleaned`, `workspace.default.transactions_cleaned`, `workspace.default.oil_cleaned`.

**Caution:** the final write cell uses `overwrite` and will replace the *current contents* of these four practice tables; it does not overwrite the source CSVs.

## 1. Set up and read the four required datasets

Define the Spark imports, the Volume location and a reusable reader once. `holidays_events.csv` is intentionally deferred: holiday coverage and locale rules need separate modeling. The Kaggle `test.csv` is reserved for the forecasting phase.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

RAW_VOLUME = "/Volumes/workspace/default/retail_raw"
TABLE_SCHEMA = "workspace.default"

def read_raw_csv(filename):
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(f"{RAW_VOLUME}/{filename}")
    )

df_sales = read_raw_csv("train.csv")
df_stores = read_raw_csv("stores.csv")
df_transactions = read_raw_csv("transactions.csv")
df_oil = read_raw_csv("oil.csv")

## 2. Reusable NULL check

Build a single aggregation for all columns of a DataFrame. Unlike running `filter(...).count()` separately for each column, this computes all NULL counts together. The `for` loop **builds expressions**; `show()` triggers the Spark job.

In [0]:
def show_missing_counts(df):
    expressions = []
    for column_name in df.columns:
        null_count = F.count(
            F.when(F.col(column_name).isNull(), 1)
        ).alias(column_name)
        expressions.append(null_count)

    df.select(*expressions).show(truncate=False)

## 3. Validate sales data

**Expected key:** `date` + `store_nbr` + `family`. Check NULLs, duplicated keys, and the numeric range. Do not delete high sales quantities simply because they are extreme: a high value is not automatically erroneous.

In [0]:
# Missing values across original six columns
show_missing_counts(df_sales)

# Duplicate business keys
sales_duplicates = (
    df_sales
    .groupBy("date", "store_nbr", "family")
    .count()
    .filter(F.col("count") > 1)
)
print("Duplicate sales keys:", sales_duplicates.count())

# Basic numeric validity check
df_sales.agg(
    F.min("sales").alias("min_sales"),
    F.max("sales").alias("max_sales"),
    F.min("onpromotion").alias("min_onpromotion"),
    F.max("onpromotion").alias("max_onpromotion")
).show()

# Inspect extreme values without removing them
df_sales.orderBy(F.desc("sales")).show(5, truncate=False)

## 4. Inspect date coverage and missing calendar days

A date that is absent from the entire dataset differs from a NULL `date` value inside a row. Compare observed dates to a complete calendar from the minimum to the maximum date. This does **not** establish that every store–family combination has full coverage.

In [0]:
date_summary = df_sales.agg(
    F.min("date").alias("start_date"),
    F.max("date").alias("end_date"),
    F.countDistinct("date").alias("observed_days")
)

summary = date_summary.first()
calendar_days = (summary["end_date"] - summary["start_date"]).days + 1

print("Start date:", summary["start_date"])
print("End date:", summary["end_date"])
print("Calendar days:", calendar_days)
print("Observed days:", summary["observed_days"])
print("Missing calendar days:", calendar_days - summary["observed_days"])

all_dates = date_summary.select(
    F.explode(F.sequence("start_date", "end_date")).alias("date")
)
observed_dates = df_sales.select("date").distinct()
missing_dates = all_dates.join(observed_dates, on="date", how="left_anti")
missing_dates.orderBy("date").show(truncate=False)

## 5. Add features to the sales dataset

Preserve all original columns and rows, then add `year`, `month` and `day_of_week` (Spark: Sunday = 1, Saturday = 7). The transformation produces a new DataFrame and does not modify `train.csv`.

In [0]:
df_sales_cleaned = (
    df_sales
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("day_of_week", F.dayofweek("date"))
)

df_sales_cleaned.printSchema()
df_sales_cleaned.show(5, truncate=False)

## 6. Validate store reference data

Each `store_nbr` should identify one store. Verify NULLs in all five columns, then check key uniqueness. No changes are applied if these checks pass.

In [0]:
show_missing_counts(df_stores)

store_duplicates = (
    df_stores
    .groupBy("store_nbr")
    .count()
    .filter(F.col("count") > 1)
)
print("Duplicate store IDs:", store_duplicates.count())
print("Total stores:", df_stores.count())

# No row-level edits required after validation.
df_stores_cleaned = df_stores

## 7. Validate transaction data

The expected key is `date` + `store_nbr`. Calculate NULL counts, duplicated keys and observed transaction-count range. A missing store-date row does not necessarily mean zero transactions.

In [0]:
show_missing_counts(df_transactions)

transaction_duplicates = (
    df_transactions
    .groupBy("date", "store_nbr")
    .count()
    .filter(F.col("count") > 1)
)
print("Duplicate transaction keys:", transaction_duplicates.count())

df_transactions.agg(
    F.min("transactions").alias("min_transactions"),
    F.max("transactions").alias("max_transactions"),
    F.avg("transactions").alias("avg_transactions")
).show()

# No row-level edits required after validation.
df_transactions_cleaned = df_transactions

## 8. Inspect NULL oil prices

The oil series has 1,218 rows and (in this dataset) 43 missing prices. Inspect the first missing dates before choosing a fill strategy. A missing quote does not mean a price of zero.

In [0]:
show_missing_counts(df_oil)

missing_oil_dates = (
    df_oil
    .filter(F.col("dcoilwtico").isNull())
    .orderBy("date")
)
missing_oil_dates.show(15, truncate=False)

## 9. Forward-fill the known oil price values

Use the latest known price **at or before** the current observation, avoiding future-looking data. The first date (`2013-01-01`) remains NULL because no earlier price is available. This fills only existing rows; it does not create weekend or holiday dates.

Spark may warn that an unpartitioned window runs on a single partition. That is acceptable for this **1,218-row single-series** dataset. In a multi-store series, use an appropriate `partitionBy` instead.

In [0]:
oil_window = (
    Window
    .orderBy("date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_oil_cleaned = df_oil.withColumn(
    "dcoilwtico",
    F.last("dcoilwtico", ignorenulls=True).over(oil_window)
)

print("Oil rows before:", df_oil.count())
print("Oil rows after:", df_oil_cleaned.count())
print("Remaining NULL oil prices:",
      df_oil_cleaned.filter(F.col("dcoilwtico").isNull()).count())

# Illustrate one forward-filled holiday price
print("Original:")
df_oil.filter(F.col("date") == "2013-01-21").show()
print("Forward filled:")
df_oil_cleaned.filter(F.col("date") == "2013-01-21").show()

## 10. Check store reference integrity

Compare the distinct store IDs used in sales and transactions with the store reference. A `left_anti` join finds IDs present on the left but missing on the right. Do not assume this validates coverage for *every date*.

In [0]:
valid_store_ids = df_stores_cleaned.select("store_nbr").distinct()

sales_store_ids = df_sales_cleaned.select("store_nbr").distinct()
unmatched_sales_ids = sales_store_ids.join(
    valid_store_ids, on="store_nbr", how="left_anti"
)

transaction_store_ids = df_transactions_cleaned.select("store_nbr").distinct()
unmatched_transaction_ids = transaction_store_ids.join(
    valid_store_ids, on="store_nbr", how="left_anti"
)

print("Unmatched sales store IDs:", unmatched_sales_ids.count())
print("Unmatched transaction store IDs:", unmatched_transaction_ids.count())

## 11. Persist validated / transformed datasets as managed Delta tables

All four tables retain their original row counts. `sales_cleaned` gains three date columns; `oil_cleaned` has forward-filled prices with the leading NULL retained; store and transaction data are validated and stored unchanged.

**Important:** `mode("overwrite")` replaces any existing content in these specific managed tables when this cell is rerun. It does not change the original Volume CSVs. Make sure this is your personal practice Catalog/Schema before running.

In [0]:
# Save the four prepared datasets
prepared_tables = [
    ("sales_cleaned", df_sales_cleaned),
    ("stores_cleaned", df_stores_cleaned),
    ("transactions_cleaned", df_transactions_cleaned),
    ("oil_cleaned", df_oil_cleaned),
]

for table_name, dataframe in prepared_tables:
    full_table_name = f"{TABLE_SCHEMA}.{table_name}"
    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(full_table_name)
    )
    print("Saved:", full_table_name)

## 12. Verify the persisted Delta tables

Read each table from the Catalog—not from the in-memory variable—to verify that the write succeeded. A reopened notebook can use `spark.table("workspace.default.sales_cleaned")` directly.

In [0]:
for table_name in ["sales_cleaned", "stores_cleaned", "transactions_cleaned", "oil_cleaned"]:
    full_table_name = f"{TABLE_SCHEMA}.{table_name}"
    saved_rows = spark.table(full_table_name).count()
    print(f"{full_table_name}: {saved_rows:,} rows")

print("Remaining NULL values in saved oil table:")
spark.table(f"{TABLE_SCHEMA}.oil_cleaned").filter(
    F.col("dcoilwtico").isNull()
).show()

## Summary / limitations

- Raw CSVs remain unchanged. All Delta tables store the complete rows from their respective source files.
- No NULL, duplicate business keys or negative values were found in the checked sales/transactions fields during the original exercise. The notebook recomputes those observations when rerun.
- High sales values are retained rather than arbitrarily removed; they require context before being labeled anomalies.
- The 4 absent Christmas dates were identified but **not** imputed in the sales dataset.
- Forward fill only handles NULL oil prices on existing rows; the first day's NULL remains.
- `holidays_events.csv` has not been cleaned or joined. Locale-sensitive analysis belongs in a later notebook.
- Yearly summaries and visualizations belong in **`03_Sales_Analysis`**, not a data-loading or cleaning notebook.